# Feature Correlation Refiner (FCR)

The FCR module models local structural correlations between encoder and decoder features and adaptively fuses the resulting structure-enhanced representation with the encoder feature.


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F


## 1. Q/K/V Projection

The decoder feature is projected to the query representation, while the encoder feature is projected to the key and value representations through independent $1\times1$ convolutions.


In [ ]:
self.conv_query = nn.Conv2d(in_channels, in_channels // 8, 1)
self.conv_key = nn.Conv2d(in_channels, in_channels // 8, 1)
self.conv_value = nn.Conv2d(in_channels, in_channels, 1)

Q = self.conv_query(F_de)
K = self.conv_key(F_en)
V = self.conv_value(F_en)


## 2. Local Window Partition

The projected features are partitioned into fixed-size local windows. The source implementation supports `overlap`, `pad_to_divisible`, and `no_pad` modes, with `overlap` used as the default mode.


In [ ]:
assert pad_mode in ('overlap', 'pad_to_divisible', 'no_pad')
self.pad_mode = pad_mode

if self.pad_mode == 'no_pad':
    stride = win
    pad_h = pad_w = 0
elif self.pad_mode == 'pad_to_divisible':
    pad_h = (win - (H_orig % win)) % win
    pad_w = (win - (W_orig % win)) % win
    if pad_h != 0 or pad_w != 0:
        pad = (0, pad_w, 0, pad_h)
        Q = F.pad(Q, pad, mode='reflect')
        K = F.pad(K, pad, mode='reflect')
        V = F.pad(V, pad, mode='reflect')
    stride = win
else:
    stride = max(1, win // 2)
    H2 = ((H_orig - win) // stride + 1) * stride + win
    W2 = ((W_orig - win) // stride + 1) * stride + win
    pad_h = max(0, H2 - H_orig)
    pad_w = max(0, W2 - W_orig)
    if pad_h != 0 or pad_w != 0:
        pad = (0, pad_w, 0, pad_h)
        Q = F.pad(Q, pad, mode='reflect')
        K = F.pad(K, pad, mode='reflect')
        V = F.pad(V, pad, mode='reflect')


## 3. Window Serialization

The local windows are serialized using `unfold`, followed by dimension permutation to organize the query, key, and value tensors for local attention modeling.


In [ ]:
_, Cq, H2, W2 = Q.shape
S = win * win

Q_unf = F.unfold(Q, kernel_size=win, stride=stride)
K_unf = F.unfold(K, kernel_size=win, stride=stride)
V_unf = F.unfold(V, kernel_size=win, stride=stride)

N = Q_unf.shape[-1]

Q_unf = Q_unf.view(B, Cq, S, N).permute(0, 3, 1, 2).contiguous()
K_unf = K_unf.view(B, Cq, S, N).permute(0, 3, 1, 2).contiguous()
V_unf = V_unf.view(B, C, S, N).permute(0, 3, 1, 2).contiguous()


## 4. Local Structural Attention

Structural correlations within each local window are modeled by scaled dot-product attention:

$$O_{\mathrm{Struct}}=\mathrm{Softmax}\left(\frac{QK^{T}}{\sqrt{d_k}}\right)V.$$


## 5. Spatial Restoration

The window-level structural features are mapped back to their corresponding spatial locations using `fold`. For overlapping windows, the accumulated responses are normalized before restoring the original spatial size.


In [ ]:
out = out.permute(0, 2, 3, 1).contiguous().view(B, C * S, N)

out_fold = F.fold(out, output_size=(H2, W2), kernel_size=win, stride=stride)

if stride != win:
    ones = torch.ones(B, C, H2, W2, device=out_fold.device, dtype=out_fold.dtype)
    ones_unf = F.unfold(ones, kernel_size=win, stride=stride)
    norm = F.fold(ones_unf, output_size=(H2, W2), kernel_size=win, stride=stride)
    out_fold = out_fold / (norm + 1e-6)

if H2 != H_orig or W2 != W_orig:
    out_fold = out_fold[:, :, :H_orig, :W_orig]

return self.conv_out(out_fold)


## 6. Gating Weight

The structure-enhanced feature and the encoder feature are concatenated to generate the pixel-wise gating weight:

$$G_1=C_1+C_2\cdot\sigma\left(W_g*[F_{en};O_{\mathrm{Struct}}]\right).$$


In [ ]:
self.gate_conv = nn.Conv2d(in_channels * 2, 1, 1)
self.sigmoid = nn.Sigmoid()
self.C1 = C1
self.C2 = C2

o_struct = self.o_struct(F_en, F_de)

gate_input = torch.cat([F_en, o_struct], dim=1)
gate_weight = self.sigmoid(self.gate_conv(gate_input))

gate_weight = self.C1 + self.C2 * gate_weight


## 7. Gated Feature Fusion

The final FCR feature is obtained by weighted fusion:

$$F_{\mathrm{FCR}}=(1-G_1)\odot F_{en}+G_1\odot O_{\mathrm{Struct}}.$$


In [ ]:
output = (1 - gate_weight) * F_en + gate_weight * o_struct
return self.channel_adjust(output)
